# Mutual Fund Analytics — Fund Performance Analytics

**Ticket:** Fund Performance Analytics
**Deliverables:** `Performance_Analytics.ipynb`, `fund_scorecard.csv`, `alpha_beta.csv`, benchmark comparison chart PNG.

Reads from `data/processed/` (Day 2 cleaned data). Computes daily returns, CAGR, Sharpe, Sortino,
Alpha/Beta vs Nifty 100, Maximum Drawdown, a composite 0-100 Fund Scorecard, and a benchmark
comparison chart for the top 5 funds.


## Setup — imports and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

PROCESSED_DIR = "data/processed"
CHARTS_DIR = "reports/performance_charts"
os.makedirs(CHARTS_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")

RISK_FREE_RATE_ANNUAL = 0.065   # RBI repo rate proxy
TRADING_DAYS = 252

fund_master = pd.read_csv(f"{PROCESSED_DIR}/01_fund_master.csv")
nav_history = pd.read_csv(f"{PROCESSED_DIR}/02_nav_history.csv", parse_dates=["date"])
scheme_performance = pd.read_csv(f"{PROCESSED_DIR}/07_scheme_performance.csv")
benchmark_indices = pd.read_csv(f"{PROCESSED_DIR}/10_benchmark_indices.csv", parse_dates=["date"])

print("Datasets loaded:")
print(f"  fund_master: {fund_master.shape}")
print(f"  nav_history: {nav_history.shape}")
print(f"  scheme_performance: {scheme_performance.shape}")
print(f"  benchmark_indices: {benchmark_indices.shape}")
print()
print("Available benchmark index names:", benchmark_indices["index_name"].unique().tolist())


## Task 1 — Daily Returns

`daily_return = nav_t / nav_t-1 - 1` for all 40 schemes. Validate the distribution looks reasonable
(mostly small values clustered near zero, no absurd outliers).

In [ ]:
nav_wide = nav_history.pivot(index="date", columns="amfi_code", values="nav").sort_index()
daily_returns = nav_wide.pct_change().dropna(how="all")

print(f"Daily returns computed for {daily_returns.shape[1]} funds across {daily_returns.shape[0]} trading days.")
print()
print("Distribution summary (all funds combined):")
print(daily_returns.stack().describe())

# Flag anomalies: daily moves greater than +/-20% are unusual for regular mutual funds
flat_returns = daily_returns.stack()
anomalies = flat_returns[flat_returns.abs() > 0.20]
print(f"\nReturns with |daily move| > 20%: {len(anomalies)} (out of {len(flat_returns)} total data points)")

plt.figure(figsize=(10, 5))
sns.histplot(flat_returns, bins=100, kde=True)
plt.title("Distribution of Daily Returns — All Funds Combined")
plt.xlabel("Daily Return")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/01_daily_return_distribution.png", dpi=150)
plt.show()


## Task 2 — CAGR (1yr, 3yr, 5yr)

`CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`. Comparison table across all funds.

In [ ]:
def compute_cagr(nav_series: pd.Series, years: float) -> float:
    """Compute CAGR using the closest available NAV to (end_date - years) as the start point."""
    nav_series = nav_series.dropna()
    if nav_series.empty:
        return np.nan
    end_date = nav_series.index.max()
    end_nav = nav_series.loc[end_date]
    target_start_date = end_date - pd.DateOffset(years=years)
    available_before = nav_series.index[nav_series.index <= target_start_date]
    if len(available_before) == 0:
        return np.nan
    start_date = available_before.max()
    start_nav = nav_series.loc[start_date]
    actual_years = (end_date - start_date).days / 365.25
    if start_nav <= 0 or actual_years <= 0:
        return np.nan
    return (end_nav / start_nav) ** (1 / actual_years) - 1

cagr_rows = []
for code in nav_wide.columns:
    series = nav_wide[code]
    cagr_rows.append({
        "amfi_code": code,
        "cagr_1yr": compute_cagr(series, 1),
        "cagr_3yr": compute_cagr(series, 3),
        "cagr_5yr": compute_cagr(series, 5),
    })

cagr_table = pd.DataFrame(cagr_rows).merge(
    fund_master[["amfi_code", "scheme_name", "fund_house", "category"]], on="amfi_code", how="left"
)
cagr_table = cagr_table[["amfi_code", "scheme_name", "fund_house", "category", "cagr_1yr", "cagr_3yr", "cagr_5yr"]]
cagr_table[["cagr_1yr", "cagr_3yr", "cagr_5yr"]] = cagr_table[["cagr_1yr", "cagr_3yr", "cagr_5yr"]].round(4)

cagr_table.sort_values("cagr_3yr", ascending=False).head(10)


## Task 3 — Sharpe Ratio

`(Rp - Rf) / Std(Rp) x sqrt(252)`, using Rf = 6.5% annual. Ranked across all 40 funds.

In [ ]:
rf_daily = RISK_FREE_RATE_ANNUAL / TRADING_DAYS

sharpe_rows = []
for code in daily_returns.columns:
    r = daily_returns[code].dropna()
    if len(r) < 2 or r.std() == 0:
        sharpe = np.nan
    else:
        sharpe = (r.mean() - rf_daily) / r.std() * np.sqrt(TRADING_DAYS)
    sharpe_rows.append({"amfi_code": code, "sharpe_ratio": sharpe})

sharpe_table = pd.DataFrame(sharpe_rows).merge(
    fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left"
)
sharpe_table["sharpe_ratio"] = sharpe_table["sharpe_ratio"].round(3)
sharpe_table["sharpe_rank"] = sharpe_table["sharpe_ratio"].rank(ascending=False, method="min").astype("Int64")
sharpe_table = sharpe_table.sort_values("sharpe_rank")

sharpe_table.head(10)


## Task 4 — Sortino Ratio

Same formula as Sharpe, but the denominator uses only downside standard deviation
(standard deviation of negative-return days only).

In [ ]:
sortino_rows = []
for code in daily_returns.columns:
    r = daily_returns[code].dropna()
    downside = r[r < 0]
    if len(r) < 2 or len(downside) == 0 or downside.std() == 0:
        sortino = np.nan
    else:
        sortino = (r.mean() - rf_daily) / downside.std() * np.sqrt(TRADING_DAYS)
    sortino_rows.append({"amfi_code": code, "sortino_ratio": sortino})

sortino_table = pd.DataFrame(sortino_rows).merge(
    fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left"
)
sortino_table["sortino_ratio"] = sortino_table["sortino_ratio"].round(3)
sortino_table["sortino_rank"] = sortino_table["sortino_ratio"].rank(ascending=False, method="min").astype("Int64")
sortino_table = sortino_table.sort_values("sortino_rank")

sortino_table.head(10)


## Task 5 — Alpha and Beta (vs Nifty 100)

OLS regression of each fund's daily returns on Nifty 100 daily returns using `scipy.stats.linregress`.
`Alpha = intercept x 252` (annualised), `Beta = slope`.

In [ ]:
# Detect the Nifty 100 index name dynamically (handles naming variations like "NIFTY100" / "NIFTY 100")
available_indices = benchmark_indices["index_name"].unique().tolist()
nifty100_name = next((n for n in available_indices if "100" in n.upper()), None)
nifty50_name = next((n for n in available_indices if "50" in n.upper() and "100" not in n.upper()), None)

print(f"Using '{nifty100_name}' as Nifty 100 benchmark, '{nifty50_name}' as Nifty 50 benchmark.")

benchmark_100 = (
    benchmark_indices[benchmark_indices["index_name"] == nifty100_name]
    .set_index("date")["close_value"].sort_index()
)
benchmark_100_returns = benchmark_100.pct_change().dropna()

alpha_beta_rows = []
for code in daily_returns.columns:
    fund_r = daily_returns[code].dropna()
    aligned = pd.concat([fund_r, benchmark_100_returns], axis=1, join="inner")
    aligned.columns = ["fund", "benchmark"]
    aligned = aligned.dropna()
    if len(aligned) < 10:
        alpha, beta, r_value, p_value = np.nan, np.nan, np.nan, np.nan
    else:
        slope, intercept, r_value, p_value, std_err = stats.linregress(aligned["benchmark"], aligned["fund"])
        beta = slope
        alpha = intercept * TRADING_DAYS
    alpha_beta_rows.append({
        "amfi_code": code, "alpha": alpha, "beta": beta,
        "r_squared": r_value ** 2 if pd.notna(r_value) else np.nan, "p_value": p_value,
    })

alpha_beta_table = pd.DataFrame(alpha_beta_rows).merge(
    fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left"
)
alpha_beta_table[["alpha", "beta", "r_squared", "p_value"]] = alpha_beta_table[
    ["alpha", "beta", "r_squared", "p_value"]
].round(4)
alpha_beta_table["alpha_rank"] = alpha_beta_table["alpha"].rank(ascending=False, method="min").astype("Int64")
alpha_beta_table = alpha_beta_table.sort_values("alpha_rank")

alpha_beta_table.to_csv("alpha_beta.csv", index=False)
print(f"Saved alpha_beta.csv ({len(alpha_beta_table)} funds)")
alpha_beta_table.head(10)


## Task 6 — Maximum Drawdown

`min(NAV / running_max - 1)` for each fund. Also finds the worst drawdown date range (peak to trough).

In [ ]:
drawdown_rows = []
for code in nav_wide.columns:
    series = nav_wide[code].dropna()
    if series.empty:
        continue
    running_max = series.cummax()
    drawdown = series / running_max - 1
    trough_date = drawdown.idxmin()
    max_dd = drawdown.min()
    peak_date = series.loc[:trough_date].idxmax()
    drawdown_rows.append({
        "amfi_code": code,
        "max_drawdown": max_dd,
        "peak_date": peak_date,
        "trough_date": trough_date,
    })

drawdown_table = pd.DataFrame(drawdown_rows).merge(
    fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left"
)
drawdown_table["max_drawdown"] = drawdown_table["max_drawdown"].round(4)
drawdown_table["dd_rank"] = drawdown_table["max_drawdown"].rank(ascending=False, method="min").astype("Int64")
drawdown_table = drawdown_table.sort_values("max_drawdown")  # worst first

drawdown_table.head(10)


## Task 7 — Fund Scorecard (0–100)

Composite score: 30% x 3yr return rank + 25% x Sharpe rank + 20% x Alpha rank
+ 15% x expense ratio rank (inverse, lower expense = better) + 10% x max drawdown rank (inverse, smaller drawdown = better).

In [ ]:
scorecard = (
    cagr_table[["amfi_code", "scheme_name", "fund_house", "category", "cagr_3yr"]]
    .merge(sharpe_table[["amfi_code", "sharpe_ratio"]], on="amfi_code", how="left")
    .merge(alpha_beta_table[["amfi_code", "alpha", "beta"]], on="amfi_code", how="left")
    .merge(fund_master[["amfi_code", "expense_ratio_pct"]], on="amfi_code", how="left")
    .merge(drawdown_table[["amfi_code", "max_drawdown"]], on="amfi_code", how="left")
)

# Percentile rank each metric 0-100 (higher percentile = better).
# For expense_ratio, lower is better, so rank ascending=False (inverse) gives the lowest expense the highest percentile.
scorecard["pct_return_3yr"] = scorecard["cagr_3yr"].rank(pct=True, ascending=True) * 100
scorecard["pct_sharpe"] = scorecard["sharpe_ratio"].rank(pct=True, ascending=True) * 100
scorecard["pct_alpha"] = scorecard["alpha"].rank(pct=True, ascending=True) * 100
scorecard["pct_expense_inverse"] = scorecard["expense_ratio_pct"].rank(pct=True, ascending=False) * 100
scorecard["pct_maxdd_inverse"] = scorecard["max_drawdown"].rank(pct=True, ascending=True) * 100  # less negative = higher rank already

scorecard["fund_score"] = (
    0.30 * scorecard["pct_return_3yr"]
    + 0.25 * scorecard["pct_sharpe"]
    + 0.20 * scorecard["pct_alpha"]
    + 0.15 * scorecard["pct_expense_inverse"]
    + 0.10 * scorecard["pct_maxdd_inverse"]
).round(2)

scorecard["overall_rank"] = scorecard["fund_score"].rank(ascending=False, method="min").astype("Int64")
scorecard = scorecard.sort_values("overall_rank")

final_scorecard = scorecard[[
    "overall_rank", "amfi_code", "scheme_name", "fund_house", "category",
    "cagr_3yr", "sharpe_ratio", "alpha", "expense_ratio_pct", "max_drawdown", "fund_score"
]]

final_scorecard.to_csv("fund_scorecard.csv", index=False)
print(f"Saved fund_scorecard.csv ({len(final_scorecard)} funds)")
final_scorecard.head(10)


## Task 8 — Benchmark Comparison Chart

Top 5 funds (by scorecard) plotted against Nifty 50 and Nifty 100 over the last 3 years,
normalised to a common starting base of 100. Tracking error = `std(fund_return - benchmark_return) x sqrt(252)`,
measured against Nifty 100.

In [ ]:
top5_codes = final_scorecard.head(5)["amfi_code"].tolist()
top5_names = final_scorecard.head(5).set_index("amfi_code")["scheme_name"].to_dict()

end_date = nav_wide.index.max()
start_date = end_date - pd.DateOffset(years=3)

# Normalise fund NAVs to start at 100
plt.figure(figsize=(14, 8))
for code in top5_codes:
    series = nav_wide[code].loc[start_date:end_date].dropna()
    normalised = series / series.iloc[0] * 100
    plt.plot(normalised.index, normalised.values, label=top5_names.get(code, str(code)))

if nifty50_name:
    b50 = benchmark_indices[benchmark_indices["index_name"] == nifty50_name].set_index("date")["close_value"].sort_index()
    b50 = b50.loc[start_date:end_date]
    if not b50.empty:
        plt.plot(b50.index, b50 / b50.iloc[0] * 100, "--", color="black", linewidth=2, label="Nifty 50")

if nifty100_name:
    b100 = benchmark_indices[benchmark_indices["index_name"] == nifty100_name].set_index("date")["close_value"].sort_index()
    b100 = b100.loc[start_date:end_date]
    if not b100.empty:
        plt.plot(b100.index, b100 / b100.iloc[0] * 100, "--", color="gray", linewidth=2, label="Nifty 100")

plt.title("Top 5 Funds vs Nifty 50 / Nifty 100 — 3 Year Comparison (Normalised to 100)")
plt.xlabel("Date")
plt.ylabel("Normalised Value (Start = 100)")
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/08_benchmark_comparison.png", dpi=150)
plt.show()
print("Saved 08_benchmark_comparison.png")

# Tracking error vs Nifty 100
print("\nTracking error (annualised) vs Nifty 100:")
for code in top5_codes:
    fund_r = daily_returns[code].loc[start_date:end_date].dropna()
    aligned = pd.concat([fund_r, benchmark_100_returns], axis=1, join="inner").dropna()
    aligned.columns = ["fund", "benchmark"]
    tracking_error = (aligned["fund"] - aligned["benchmark"]).std() * np.sqrt(TRADING_DAYS)
    print(f"  {top5_names.get(code, code)}: {tracking_error:.4f}")
